# Multibanding Mismatch Validation

This notebook confirms that the `MultibandedFrequencyDomain` preserves
waveform fidelity relative to the full-resolution base domain.

## Method

1. Generate TDI-A/E waveforms on the base `UniformFrequencyDomain` (`bbhx_gen_base`).
2. Generate the same waveforms **directly** on the `MultibandedFrequencyDomain`
   (`bbhx_gen_mbd`) — BBHx evaluates at the MBD sample frequencies by point evaluation,
   exactly as DINGO does for FD approximants.
3. Sample the base-domain waveform at the MBD frequencies (nearest-neighbour) to bring
   both representations onto the same grid.
4. Compute the standard overlap/mismatch on the MBD grid, weighted by the LISA PSD
   and per-bin Δf from `mbd._delta_f`.
5. Sweep over sources drawn from the MBHB prior to obtain a mismatch distribution.

**Run this notebook in the cluster environment** after bootstrapping with
`misc_scripts/lisa_cluster_bootstrap.sh` or equivalent SLURM setup.

In [ ]:
from pathlib import Path
import sys
import math
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / 'misc_scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from misc_scripts.compare_lisa_bbhx_lisabeta import (
    environment_report,
    _require_waveform_stack,
    DEFAULT_LISA_SETTINGS,
    PYCONSTANTS_YRSID_SI,
)

print(environment_report())

In [ ]:
_require_waveform_stack()

from dingo.gw.waveform_generator.waveform_generator import BBHxWaveformGenerator
from dingo.gw.domains import build_domain
from dingo.gw.domains.multibanded_frequency_domain import MultibandedFrequencyDomain
from dingo.gw.domains.uniform_frequency_domain import UniformFrequencyDomain

print('Imports OK')

## Domain setup

In [ ]:
# Base (full-resolution) uniform domain
BASE_DOMAIN_SETTINGS = dict(
    type='UniformFrequencyDomain',
    f_min=1e-4,
    f_max=1e-1,
    delta_f=5e-6,
)
base_domain = build_domain(BASE_DOMAIN_SETTINGS)
freqs_base  = np.array(base_domain.sample_frequencies)
df_base     = base_domain.delta_f
print(f'Base domain: {len(freqs_base)} bins, '
      f'{freqs_base[0]:.2e}–{freqs_base[-1]:.2e} Hz,  Δf = {df_base:.1e} Hz')

# Multibanded domain — same nodes as the training config
#   nodes = band boundaries,  delta_f doubles each band
MBD_NODES        = [1.0e-4, 1.0e-3, 5.0e-3, 2.0e-2, 1.0e-1]
MBD_DELTA_F_INIT = 5.0e-6   # Hz  (same as base_domain.delta_f — band 0 = no decimation)

mbd = MultibandedFrequencyDomain(
    nodes=MBD_NODES,
    delta_f_initial=MBD_DELTA_F_INIT,
    base_domain=BASE_DOMAIN_SETTINGS,
)
freqs_mbd = np.array(mbd.sample_frequencies)
print(f'Multibanded domain: {len(freqs_mbd)} bins across {mbd.num_bands} bands')
print(f'  Band Δf values: {[f"{x:.1e}" for x in mbd._delta_f_bands]}')
print(f'  Bins per band:  {list(mbd._num_bins_bands)}')

## Waveform generator

In [ ]:
# Two generators: base (full-resolution) and MBD (direct point evaluation).
# DINGO evaluates FD approximants like BBHx directly at the domain sample frequencies,
# so the MBD generator gives the ground-truth MBD representation.
bbhx_gen_base = BBHxWaveformGenerator(
    approximant='PhenomHM',
    domain=BASE_DOMAIN_SETTINGS,
    f_ref=1e-3,
    use_gpu=False,
    direct_response=False,
    bbhx_t_obs_start_years=0.0,
    bbhx_t_obs_end_years=0.25,
    default_t_ref_years=0.75,
    bbhx_length=2048,
    isco_cutoff=True,
)

bbhx_gen_mbd = BBHxWaveformGenerator(
    approximant='PhenomHM',
    domain=dict(
        type='MultibandedFrequencyDomain',
        nodes=MBD_NODES,
        delta_f_initial=MBD_DELTA_F_INIT,
        base_domain=BASE_DOMAIN_SETTINGS,
    ),
    f_ref=1e-3,
    use_gpu=False,
    direct_response=False,
    bbhx_t_obs_start_years=0.0,
    bbhx_t_obs_end_years=0.25,
    default_t_ref_years=0.75,
    bbhx_length=2048,
    isco_cutoff=True,
)

print('Generators initialised')
print(f'Base domain length: {len(np.array(bbhx_gen_base.domain.sample_frequencies))} bins')
print(f'MBD  domain length: {len(np.array(bbhx_gen_mbd.domain.sample_frequencies))} bins')

## LISA PSD and overlap helpers

In [ ]:
C_SI = 3e8

def lisa_psd_A(f, L=2.5e9):
    f   = np.asarray(f, dtype=float)
    S_oms = (1.5e-11)**2 * (1 + (2e-3 / np.maximum(f, 1e-10))**4)
    S_acc = (3e-15)**2 * (1 + (4e-4 / np.maximum(f, 1e-10))**2) * \
            (1 + (f / 8e-3)**4) / (2 * math.pi * np.maximum(f, 1e-10))**4
    S_link = (S_oms + 2 * S_acc) / L**2
    x = 2 * math.pi * f * L / C_SI
    return 8 * np.sin(x)**2 * (2 * (1 + np.cos(x)**2) * S_link)


def inner_product(a, b, psd, df):
    return 4 * df * np.real(np.sum(np.conj(a) * b / psd))


def mismatch(h1, h2, psd, df):
    """1 − overlap, both h1 and h2 on the SAME uniform frequency grid."""
    ov = inner_product(h1, h2, psd, df)
    ov /= math.sqrt(inner_product(h1, h1, psd, df) * inner_product(h2, h2, psd, df))
    return 1.0 - ov


psd_base = lisa_psd_A(freqs_base)
psd_base = np.where(freqs_base > 0, psd_base, np.inf)
print('PSD computed')

## Validation metric: overlap / mismatch

For each source we generate the waveform on both the base domain and the MBD domain,
then compute the standard overlap on the MBD grid:

    inner(h1, h2) = 4 · Re Σ_k  conj(h1[k]) · h2[k] / S(f_k) · Δf_k
    overlap       = |inner(h_base_nn, h_mbd)| / sqrt(inner(h_base_nn, h_base_nn) · inner(h_mbd, h_mbd))
    mismatch      = 1 − overlap

where `h_base_nn[k]` is the base-domain waveform sampled at the MBD frequency `f_k`
(nearest-neighbour), and `Δf_k = mbd._delta_f[k]` is the per-bin width on the MBD grid.

Pass criterion: median mismatch < 0.1% (1e-3) across the prior sweep.

In [ ]:
# PSD and per-bin Δf on the MBD grid (set up once, reused in prior sweep)
psd_mbd     = lisa_psd_A(freqs_mbd)
psd_mbd     = np.where(freqs_mbd > 0, psd_mbd, np.inf)
df_mbd_bins = np.array(mbd._delta_f)   # shape (n_mbd,), per-bin Δf weights


def inner_product_mbd(a, b):
    """Inner product on the MBD grid using per-bin Δf weights."""
    return 4 * np.real(np.sum(np.conj(a) * b / psd_mbd * df_mbd_bins))


def mismatch_mbd(h1, h2):
    """Standard mismatch (1 − overlap) between two waveforms on the MBD grid."""
    ov = inner_product_mbd(h1, h2)
    ov /= math.sqrt(inner_product_mbd(h1, h1) * inner_product_mbd(h2, h2))
    return 1.0 - abs(ov)


# Single-source sanity check
_params_test = dict(
    Mchirp=7e5, q=0.9, chi1=0.3, chi2=0.3,
    inc=0.5, phi=0.0, lam=0.5, beta=0.3, psi=1.2,
    dist=5000.0, geocent_time=0.25 * PYCONSTANTS_YRSID_SI,
)
wf_base = bbhx_gen_base.generate_amp_phase(_params_test)
wf_mbd  = bbhx_gen_mbd.generate_amp_phase(_params_test)

# waveform shape: (1, 3, n_freqs) — [0][ch] selects channel
h_base_chan1 = wf_base['waveform'][0][0]   # shape (n_base,)
h_mbd_chan1  = wf_mbd['waveform'][0][0]    # shape (n_mbd,)

# Sample h_base at MBD frequencies (nearest-neighbour)
idx_nn        = np.searchsorted(freqs_base, freqs_mbd).clip(0, len(freqs_base) - 1)
h_base_at_mbd = h_base_chan1[idx_nn]      # shape (n_mbd,)

mm = mismatch_mbd(h_base_at_mbd, h_mbd_chan1)
print(f'Mismatch (TDI-A, base vs MBD): {mm:.4e}')
print(f'(pass threshold < 1e-3)')
print(f'  base waveform shape: {wf_base["waveform"].shape}')
print(f'  MBD  waveform shape: {wf_mbd["waveform"].shape}')

## Visual comparison: amplitude envelope

In [ ]:
# Amplitude comparison: base domain waveform vs MBD direct evaluation
# h_base_at_mbd and h_mbd_chan1 are already on the MBD grid (from cell 10)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax = axes[0]
ax.loglog(freqs_base * 1e3, np.abs(h_base_chan1), lw=1.0, label='Base domain', color='k')
ax.scatter(freqs_mbd * 1e3, np.abs(h_mbd_chan1), s=3, color='steelblue',
           zorder=5, label='MBD direct evaluation')
ax.set_ylabel('|h(f)|  [strain/Hz]')
ax.set_title('TDI-A amplitude — base domain vs MBD direct evaluation')
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.5)

ax = axes[1]
ratio = np.abs(h_mbd_chan1) / np.maximum(np.abs(h_base_at_mbd), 1e-300)
ax.semilogx(freqs_mbd * 1e3, ratio, lw=0.0, marker='.', ms=2, color='steelblue')
ax.axhline(1.0, color='k', ls='--', lw=0.8)
for node in MBD_NODES[1:-1]:
    ax.axvline(node * 1e3, color='orange', ls=':', lw=1.0, alpha=0.7)
ax.set_ylim(0.5, 1.5)
ax.set_xlabel('Frequency [mHz]')
ax.set_ylabel('|h_mbd| / |h_base_nn| (nearest-neighbour)')
ax.set_title('Amplitude ratio at MBD bin centres (orange dotted = band boundaries)')
ax.grid(True, which='both', ls=':', alpha=0.5)

fig.tight_layout()
plt.savefig('multibanding_amplitude_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved multibanding_amplitude_comparison.png')

## Mismatch distribution over prior

In [ ]:
rng = np.random.default_rng(0)
N_SOURCES = 20

M_totals = np.exp(rng.uniform(np.log(1e5), np.log(1e6), N_SOURCES))

mm_chan1 = []
mm_chan2 = []

# Pre-compute nearest-neighbour indices (same MBD grid for all sources)
_idx_nn = np.searchsorted(freqs_base, freqs_mbd).clip(0, len(freqs_base) - 1)

for M_tot in M_totals:
    q   = rng.uniform(0.1, 1.0)
    eta = q / (1 + q)**2
    params = dict(
        Mchirp=M_tot * eta**0.6, q=q,
        chi1=rng.uniform(-0.9, 0.9),
        chi2=rng.uniform(-0.9, 0.9),
        inc=np.arccos(rng.uniform(-1, 1)),
        phi=rng.uniform(0, 2 * math.pi),
        lam=rng.uniform(0, 2 * math.pi),
        beta=np.arcsin(rng.uniform(-1, 1)),
        psi=rng.uniform(0, math.pi),
        dist=rng.uniform(500, 20000),
        geocent_time=0.25 * PYCONSTANTS_YRSID_SI,
    )
    try:
        wf_b = bbhx_gen_base.generate_amp_phase(params)
        wf_m = bbhx_gen_mbd.generate_amp_phase(params)
        # waveform shape: (1, 3, n_freqs) — [0][ch] selects channel
        for ch_idx, mm_list in [(0, mm_chan1), (1, mm_chan2)]:
            h_b      = wf_b['waveform'][0][ch_idx]   # base domain
            h_m      = wf_m['waveform'][0][ch_idx]   # MBD direct evaluation
            h_b_nn   = h_b[_idx_nn]                  # sample at MBD freqs
            mm_list.append(mismatch_mbd(h_b_nn, h_m))
    except Exception as e:
        print(f'  M_tot={M_tot:.2e}: {e}')

mm_chan1 = np.array(mm_chan1)
mm_chan2 = np.array(mm_chan2)

print(f'TDI-A mismatch: median={np.median(mm_chan1):.3e},  '
      f'90th pct={np.percentile(mm_chan1, 90):.3e},  '
      f'max={mm_chan1.max():.3e}')
print(f'TDI-E mismatch: median={np.median(mm_chan2):.3e},  '
      f'90th pct={np.percentile(mm_chan2, 90):.3e},  '
      f'max={mm_chan2.max():.3e}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

bins = np.logspace(np.log10(1e-6), np.log10(0.1), 30)
ax.hist(mm_chan1, bins=bins, alpha=0.6, label='TDI-A', color='steelblue')
ax.hist(mm_chan2, bins=bins, alpha=0.6, label='TDI-E', color='darkorange')
ax.axvline(1e-3, color='red', ls='--', lw=1.2, label='0.1% threshold')
ax.set_xscale('log')
ax.set_xlabel('Mismatch  (1 − overlap)')
ax.set_ylabel('Count')
ax.set_title(f'Multibanding mismatch distribution ({N_SOURCES} sources)')
ax.legend()
ax.grid(True, which='both', ls=':', alpha=0.5)
fig.tight_layout()
plt.savefig('multibanding_mismatch_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved multibanding_mismatch_distribution.png')

## Sensitivity to node placement

Try a coarser multibanding scheme and compare mismatch, to understand the trade-off
between compression ratio and fidelity.

In [ ]:
MBD_NODES_COARSE        = [1.0e-4, 2.0e-3, 2.0e-2, 1.0e-1]
MBD_DELTA_F_INIT_COARSE = 2.0e-5

mbd_coarse = MultibandedFrequencyDomain(
    nodes=MBD_NODES_COARSE,
    delta_f_initial=MBD_DELTA_F_INIT_COARSE,
    base_domain=BASE_DOMAIN_SETTINGS,
)
freqs_mbd_coarse = np.array(mbd_coarse.sample_frequencies)
print(f'Coarse MBD: {len(freqs_mbd_coarse)} bins across {mbd_coarse.num_bands} bands')
print(f'  Bins per band: {list(mbd_coarse._num_bins_bands)}')

# Generator that evaluates directly at coarse MBD frequencies
bbhx_gen_mbd_coarse = BBHxWaveformGenerator(
    approximant='PhenomHM',
    domain=dict(
        type='MultibandedFrequencyDomain',
        nodes=MBD_NODES_COARSE,
        delta_f_initial=MBD_DELTA_F_INIT_COARSE,
        base_domain=BASE_DOMAIN_SETTINGS,
    ),
    f_ref=1e-3,
    use_gpu=False,
    direct_response=False,
    bbhx_t_obs_start_years=0.0,
    bbhx_t_obs_end_years=0.25,
    default_t_ref_years=0.75,
    bbhx_length=2048,
    isco_cutoff=True,
)

psd_mbd_coarse     = lisa_psd_A(freqs_mbd_coarse)
psd_mbd_coarse     = np.where(freqs_mbd_coarse > 0, psd_mbd_coarse, np.inf)
df_mbd_coarse_bins = np.array(mbd_coarse._delta_f)


def inner_product_mbd_coarse(a, b):
    return 4 * np.real(np.sum(np.conj(a) * b / psd_mbd_coarse * df_mbd_coarse_bins))


def mismatch_mbd_coarse(h1, h2):
    ov = inner_product_mbd_coarse(h1, h2)
    ov /= math.sqrt(inner_product_mbd_coarse(h1, h1) * inner_product_mbd_coarse(h2, h2))
    return 1.0 - abs(ov)


# Nearest-neighbour indices for coarse MBD
_idx_nn_coarse = np.searchsorted(freqs_base, freqs_mbd_coarse).clip(0, len(freqs_base) - 1)

wf_mbd_c       = bbhx_gen_mbd_coarse.generate_amp_phase(_params_test)
h_mbd_c_chan1  = wf_mbd_c['waveform'][0][0]
h_base_at_mbd_c = h_base_chan1[_idx_nn_coarse]

mm_fine   = mismatch_mbd(h_base_at_mbd, h_mbd_chan1)
mm_coarse = mismatch_mbd_coarse(h_base_at_mbd_c, h_mbd_c_chan1)

print(f'Fine   scheme mismatch (TDI-A): {mm_fine:.4e}  ({len(freqs_mbd)} bins)')
print(f'Coarse scheme mismatch (TDI-A): {mm_coarse:.4e}  ({len(freqs_mbd_coarse)} bins)')

## Summary

- If median mismatch is < 0.1% (1e-3) with the chosen multiband nodes, the
  configuration is acceptable for training.
- If mismatch is too high (> 0.1%), either reduce `delta_f_initial` (finer first band)
  or adjust node positions to avoid sharp band transitions near the signal peak.
- The coarser scheme shows how aggressively we can compress before fidelity degrades.

Adjust `MBD_NODES` and `MBD_DELTA_F_INIT` in `waveform_dataset_settings.yaml` based
on these results.